# 241Fa18049 - Diabetes Prediction Project
MLOps / Machine Learning notebook using `data/raw/diabetes.csv`.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


## 1. Load Dataset


In [ ]:
df = pd.read_csv('data/raw/diabetes.csv')
print(df.shape)
df.head()


## 2. Data Understanding


In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nTarget distribution:')
print(df['Outcome'].value_counts())


## 3. Preprocessing
Zeros in Glucose, BloodPressure, SkinThickness, Insulin and BMI are treated as missing because zero is not a meaningful physiological measurement for these fields.


In [ ]:
TARGET='Outcome'
FEATURES=['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
ZERO_AS_MISSING=['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
work=df.drop_duplicates().copy()
work[ZERO_AS_MISSING]=work[ZERO_AS_MISSING].replace(0,np.nan)
X=work[FEATURES]
y=work[TARGET].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
preprocessor=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
X_train_final=preprocessor.fit_transform(X_train)
X_test_final=preprocessor.transform(X_test)
print(X_train_final.shape, X_test_final.shape)


## 4. Train Random Forest


In [ ]:
model=RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)
model.fit(X_train_final,y_train)
print('Model trained successfully')


## 5. Evaluation


In [ ]:
pred=model.predict(X_test_final)
prob=model.predict_proba(X_test_final)[:,1]
print(f'Accuracy : {accuracy_score(y_test,pred):.4f}')
print(f'Precision: {precision_score(y_test,pred,zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test,pred,zero_division=0):.4f}')
print(f'F1 Score : {f1_score(y_test,pred,zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test,prob):.4f}')
print('Confusion Matrix:\n',confusion_matrix(y_test,pred))


## 6. Unseen Prediction


In [ ]:
sample=pd.DataFrame([{'Pregnancies':2,'Glucose':120,'BloodPressure':70,'SkinThickness':25,'Insulin':100,'BMI':28.5,'DiabetesPedigreeFunction':0.35,'Age':30}])
sample_final=preprocessor.transform(sample)
p=model.predict(sample_final)[0]
probability=model.predict_proba(sample_final)[0,1]
print('Predicted Outcome:', int(p))
print(f'Diabetes Probability: {probability*100:.2f}%')
